# LLM-based metadata extraction from Marker-PDF markdown outputs.

In [ ]:
# Import libraries
import json
import os
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from openai import OpenAI

In [1]:
import openai
import sys

print(f"Versión de OpenAI: {openai.__version__}")
print(f"Ruta del ejecutable: {sys.executable}")

Versión de OpenAI: 2.15.0
Ruta del ejecutable: /Users/aramirezm/opt/anaconda3/envs/openai_env/bin/python


In [2]:
# Prompt / schema
SYSTEM_MSG = (
    "You are a precise metadata extractor for academic papers. "
    "Return ONLY valid JSON. No prose, no markdown, no extra keys."
)

# Single task that returns everything at once (best for cost + consistency)
# Important: enforce empty strings/lists and booleans; never 'not found' text.
EXTRACTION_INSTRUCTIONS = """\
Extract the following fields from the paper content.

Return EXACTLY this JSON schema (same keys), with these rules:
- If a field is missing/unclear, return an empty value:
  - strings: ""
  - lists: []
  - booleans: false
- Do NOT add any extra keys.
- Do NOT include explanations.

Schema:
{
  "title": "",
  "abstract": "",
  "keywords": [],
  "authors": [
    {
      "author": "",
      "affiliation": "",
      "email": "",
      "orcid": "",
      "is_corresponding": false
    }
  ]
}

Field rules:
- title: exact paper title (no journal name, no section headers like "Research papers").
- abstract: exact abstract text only (do not include Introduction spillover; stop where abstract ends).
  Do not include the "Abstract:" label.
- keywords: if explicitly listed (Keywords / Index Terms), use them; else provide 5–7 keywords that best describe the paper.
- authors: extract each author as a separate object.
  - author: name only (remove superscripts/markers like a,b,1,2,*, dagger, etc).
  - affiliation: the affiliation for that author if explicitly stated; else "".
  - email: author email if explicitly stated; else "".
  - orcid: ORCID if explicitly stated; else "".
  - is_corresponding: true only if explicitly indicated (e.g., *, "corresponding author"); else false.

Output ONLY the JSON object.
"""


In [3]:
# Marker path helpers
def marker_markdown_path(base_dir: Path, paper_id: str) -> Path:
    return base_dir / paper_id / "markdown" / f"{paper_id}_md.md"


def list_paper_ids(base_dir: Path) -> List[str]:
    out = []
    for p in base_dir.iterdir():
        if p.is_dir():
            out.append(p.name)
    # numeric sort when possible
    def key(x: str):
        try:
            return (0, int(x))
        except Exception:
            return (1, x)
    return sorted(out, key=key)

In [4]:
# Core extraction logic
def safe_json_default() -> Dict[str, Any]:
    return {
        "title": "",
        "abstract": "",
        "keywords": [],
        "authors": []
    }


def extract_one_paper(
    client: OpenAI,
    model: str,
    paper_id: str,
    md_text: str,
) -> Dict[str, Any]:
    """
    One-call extraction. Uses a stable prompt prefix (good for caching) and paper content as the variable.
    """
    # Prompt-caching friendly structure:
    # - system constant
    # - instructions constant
    # - paper content as one contiguous block reused inside the same request
    # (caching effectiveness depends on provider, but this is the best shape)
    user_content = (
        "PAPER_CONTENT_START\n"
        f"{md_text}\n"
        "PAPER_CONTENT_END\n\n"
        f"{EXTRACTION_INSTRUCTIONS}"
    )

    resp = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": user_content},
        ],
        text={
            "format": {"type": "json_object"},
            "verbosity": "medium"
        },
        # keep extraction deterministic-ish
        reasoning= {"effort": "none"},
        temperature=0.0,
    )

    # SDK returns structured output; easiest is output_text
    text = (resp.output_text or "").strip()
    if not text:
        out = safe_json_default()
        out["paper_id"] = paper_id
        return out

    data = json.loads(text)

    # Enforce schema + empty defaults (no "not found" strings)
    out = safe_json_default()
    out["paper_id"] = paper_id

    out["title"] = data.get("title") or ""
    out["abstract"] = data.get("abstract") or ""

    kw = data.get("keywords")
    out["keywords"] = kw if isinstance(kw, list) else []

    authors = data.get("authors")
    cleaned_authors: List[Dict[str, Any]] = []
    if isinstance(authors, list):
        for a in authors:
            if not isinstance(a, dict):
                continue
            cleaned_authors.append({
                "author": a.get("author") or "",
                "affiliation": a.get("affiliation") or "",
                "email": a.get("email") or "",
                "orcid": a.get("orcid") or "",
                "is_corresponding": bool(a.get("is_corresponding")) if a.get("is_corresponding") is not None else False,
            })
    out["authors"] = cleaned_authors

    return out


In [7]:
def run_pipeline(
    base_dir: Path,
    output_file: Path,
    model: str,
    *,
    sleep_s: float = 0.2,
    limit: int = 0,
) -> None:
    load_dotenv()
    client = OpenAI()

    # 1. Load previous progress or initialize a new payload
    if output_file.exists():
        try:
            payload = json.loads(output_file.read_text(encoding="utf-8"))
            print(f"📖 Loading existing progress from {output_file}")
        except json.JSONDecodeError:
            print(f"⚠️ {output_file} is corrupt. Starting from scratch.")
            payload = {
                "version": "marker_llm_v1",
                "model": model,
                "base_dir": str(base_dir.resolve()),
                "count": 0,
                "skipped": [],
                "records": [],
            }
    else:
        payload = {
            "version": "marker_llm_v1",
            "model": model,
            "base_dir": str(base_dir.resolve()),
            "count": 0,
            "skipped": [],
            "records": [],
        }

    # Create a set of already processed IDs for O(1) fast lookup
    processed_ids = {rec["paper_id"] for rec in payload["records"]}
    
    # Get the list of available paper folders
    paper_ids = list_paper_ids(base_dir)
    if limit and limit > 0:
        paper_ids = paper_ids[:limit]

    for pid in paper_ids:
        # 2. Skip if the paper has already been processed in a previous run
        if pid in processed_ids:
            # print(f"⏭️ Skipping {pid} (already exists)")
            continue

        md_path = marker_markdown_path(base_dir, pid)
        if not md_path.exists():
            payload["skipped"].append({"paper_id": pid, "reason": f"missing_markdown: {str(md_path)}"})
            # Save the skip reason to the file immediately
            output_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
            continue

        try:
            print(f"📄 Processing: {pid}...")
            md_text = md_path.read_text(encoding="utf-8", errors="replace")
            rec = extract_one_paper(
                client=client,
                model=model,
                paper_id=pid,
                md_text=md_text,
            )
            
            # 3. Update the payload and save to disk immediately
            payload["records"].append(rec)
            payload["count"] = len(payload["records"])
            
            # Write the file on every successful extraction to prevent data loss
            output_file.write_text(
                json.dumps(payload, ensure_ascii=False, indent=2), 
                encoding="utf-8"
            )
            
            # Update the set to track progress within the current loop
            processed_ids.add(pid)

        except Exception as e:
            error_msg = f"error: {type(e).__name__}: {e}"
            payload["skipped"].append({"paper_id": pid, "reason": error_msg})
            # Save the skip even on error to keep the log consistent
            output_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

        # Respect API rate limits
        time.sleep(sleep_s)

    print(f"✅ Pipeline finished. Total records in file: {payload['count']}")

In [8]:
run_pipeline(
        base_dir=Path("publications/output_marker/LLM_gpt-4o-mini"),
        output_file=Path("paper_metadata.json"),
        model="gpt-5.2",
        sleep_s=0.2,
        limit=0,
    )

📖 Loading existing progress from paper_metadata.json
📄 Processing: 318...
📄 Processing: 319...
📄 Processing: 320...
✅ Pipeline finished. Total records in file: 228
